**Validation** : Leave-One-Out (LOO) — standard pour petits datasets médicaux

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    balanced_accuracy_score, roc_auc_score
)
import xgboost as xgb
pd.set_option('display.max_columns', None)

In [3]:
DATA_PATH = "/home/justine/code/Maelle05/DyslexIA/data/data_T4/data"
files_raw = glob.glob(os.path.join(DATA_PATH, "*raw.csv"))

LABELS_PATH = "/home/justine/code/Maelle05/DyslexIA/data/dyslexia_class_label.csv"

TARGET_LEN = 2000 # A FINE TUNER
GAZE_COLS  = ['gaze_x_left', 'gaze_y_left', 'gaze_x_right', 'gaze_y_right']
SID_COL    = 'subject_id'

In [4]:
labels = pd.read_csv(LABELS_PATH)
print(f"\nDistribution : {labels['class_id'].value_counts().to_dict()}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/justine/code/Maelle05/DyslexIA/data/dyslexia_class_label.csv'

In [5]:
all_lengths = []

for f in files_raw:
    df = pd.read_csv(f)
    all_lengths.append(len(df))

print("min:", min(all_lengths))
print("max:", max(all_lengths))
print("mean:", sum(all_lengths)/len(all_lengths))

min: 11511
max: 100912
mean: 27784.114285714284


In [6]:
def to_fixed_length(signal, target_len):
    """Interpolation linéaire vers une longueur fixe."""
    x_old = np.linspace(0, 1, len(signal))
    x_new = np.linspace(0, 1, target_len)
    return interp1d(x_old, signal, kind='linear')(x_new)

In [7]:
def extract_features_selected(df, gaze_cols, target_len):
    """
    Calcule uniquement les 8 features sélectionnées.
    Indices originaux : [5, 8, 15, 42, 48, 56, 59, 69]
    Ordre GAZE_COLS : [gaze_x_left, gaze_y_left, gaze_x_right, gaze_y_right]
    """
    signals = {}
    for col in gaze_cols:
        s = to_fixed_length(df[col].values.astype(float), target_len)
        signals[col] = s

    def vel_p90(s):
        return np.percentile(np.abs(np.diff(s)), 90)

    def vel_saccade_rate(s):
        vel = np.abs(np.diff(s))
        vel_mean = np.mean(vel)
        vel_std  = np.std(vel) + 1e-8
        return np.sum(vel > vel_mean + 2 * vel_std) / len(vel)

    def vel_std_fixation_proxy(s):
        vel = np.abs(np.diff(s))
        slow_mask = vel < np.percentile(vel, 25)
        runs = np.diff(np.concatenate([[0], slow_mask.astype(int), [0]]))
        run_lengths = np.where(runs == -1)[0] - np.where(runs == 1)[0]
        return np.std(run_lengths) if len(run_lengths) > 0 else 0.0

    def temporal_iqr(s):                          # ← corrigé : IQR et non p90
        return np.percentile(s, 90) - np.percentile(s, 10)

    def vel_median(s):
        return np.median(np.abs(np.diff(s)))

    feat = np.array([
        vel_p90(signals['gaze_x_left']),               # idx 5
        vel_saccade_rate(signals['gaze_x_left']),       # idx 8
        vel_std_fixation_proxy(signals['gaze_x_left']), # idx 15
        vel_std_fixation_proxy(signals['gaze_y_left']), # idx 42
        temporal_iqr(signals['gaze_y_left']),           # idx 48  ← corrigé
        vel_median(signals['gaze_x_right']),            # idx 56
        vel_p90(signals['gaze_x_right']),               # idx 59
        vel_std_fixation_proxy(signals['gaze_x_right']),# idx 69
    ])

    return np.nan_to_num(feat, nan=0.0)

In [8]:
rows, skipped = [], []
for fpath in files_raw:
    try:
        df  = pd.read_csv(fpath)
        sid = int(df[SID_COL].iloc[0])
        label_row = labels[labels['subject_id'] == sid]
        if label_row.empty:
            skipped.append((sid, 'label manquant'))
            continue
        rows.append({'sid'     : sid,
                     'label'   : int(label_row['class_id'].values[0]),
                     'features': extract_features_selected(df, GAZE_COLS, TARGET_LEN)})
    except Exception as e:
        skipped.append((os.path.basename(fpath), str(e)))

sids = np.array([r['sid']      for r in rows])
X    = np.array([r['features'] for r in rows])
y    = np.array([r['label']    for r in rows]).astype(int)

print(f"X : {X.shape} | Classes : {np.bincount(y)}  (0=non-dys, 1=dys)")
# X : (n_sujets, 8)

X : (70, 8) | Classes : [35 35]  (0=non-dys, 1=dys)


In [9]:
param_grid = {
    'clf__max_depth'        : [2, 3, 4],
    'clf__n_estimators'     : [50, 100, 200],
    'clf__learning_rate'    : [0.01, 0.05, 0.1],
    'clf__reg_lambda'       : [0.5, 1.0, 2.0],
    'clf__colsample_bytree' : [0.6, 0.8, 1.0],
}
# Version rapide (~2 min) :
#param_grid = {'clf__max_depth':[2,3], 'clf__n_estimators':[50,100], 'clf__learning_rate':[0.05,0.1]}

loo      = LeaveOneOut()
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_true, y_pred, y_scores = [], [], []
best_params_list = []

for fold_i, (train_idx, test_idx) in enumerate(loo.split(X, y)):
    if fold_i % 10 == 0:
        print(f"Fold {fold_i+1}/{len(X)}...")

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0))
    ])

    gs = GridSearchCV(pipe, param_grid, cv=inner_cv,
                      scoring='roc_auc', n_jobs=-1, refit=True)
    gs.fit(X[train_idx], y[train_idx])

    best_params_list.append(gs.best_params_)
    y_true.append(y[test_idx][0])
    y_pred.append(gs.predict(X[test_idx])[0])
    y_scores.append(gs.predict_proba(X[test_idx])[0, 1])

y_true, y_pred, y_scores = map(np.array, [y_true, y_pred, y_scores])
print('\n✅ Nested CV terminée.')

Fold 1/70...


Fold 11/70...
Fold 21/70...
Fold 31/70...
Fold 41/70...
Fold 51/70...
Fold 61/70...

✅ Nested CV terminée.


In [11]:
print('=' * 42)
print(f'  Accuracy          : {accuracy_score(y_true, y_pred):.4f}')
print(f'  Balanced Accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}  ← principale')
print(f'  AUC-ROC           : {roc_auc_score(y_true, y_scores):.4f}  ← principale')
print(f'  Recall (dys)      : {recall_score(y_true, y_pred):.4f}  ← critique clinique')
print(f'  Precision (dys)   : {precision_score(y_true, y_pred):.4f}')
print('=' * 42)

params_df = pd.DataFrame(best_params_list)
print('\nHyperparamètres les plus souvent sélectionnés :')
print(params_df.apply(lambda col: col.value_counts().index[0]))

  Accuracy          : 0.8143
  Balanced Accuracy : 0.8143  ← principale
  AUC-ROC           : 0.8980  ← principale
  Recall (dys)      : 0.8000  ← critique clinique
  Precision (dys)   : 0.8235

Hyperparamètres les plus souvent sélectionnés :
clf__colsample_bytree     0.80
clf__learning_rate        0.01
clf__max_depth            2.00
clf__n_estimators        50.00
clf__reg_lambda           0.50
dtype: float64
